# HTRU2 Pulsar — 能量泛函 QUBO + CIM
## Parkes 射电望远镜 → 脉冲星候选体分类 → PCA 2D → Poisson PDE → $E(u)=\frac{1}{2}u^T K u - u^T F$ → CIM
单独运行本 notebook 可验证 QUBO 构建正确性，无需跑完整训练脚本

In [1]:
import io, zipfile, requests
import numpy as np
import pandas as pd
import kaiwu as kw
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings, json, os
warnings.filterwarnings('ignore')
kw.common.CheckpointManager.save_dir = '/tmp'

BIT_WIDTH = 8
OUTPUT_DIR = 'D:/QPDE/photo+kan/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('模块加载完成')

模块加载完成


In [2]:
# ===== 加载 HTRU2 脉冲星数据 =====
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00372/HTRU2.zip'
zf = zipfile.ZipFile(io.BytesIO(requests.get(url).content))
df = pd.read_csv(zf.open('HTRU_2.csv'), header=None,
    names=['mean_ip','std_ip','excess_kurtosis_ip','skewness_ip',
           'mean_dm','std_dm','excess_kurtosis_dm','skewness_dm','class'])
X_all = df.drop(columns=['class']).values.astype(float)
y_all = np.where(df['class'].values == 1, 1, -1)  # pulsar=+1, RFI=-1

rng = np.random.default_rng(42)
idx = rng.choice(len(y_all), 2000, replace=False)
X_s, y_s = X_all[idx], y_all[idx]
print(f'全部: {len(y_all)} 条, 脉冲星={(y_all==1).sum()}, RFI={(y_all==-1).sum()}')
print(f'采样: {len(y_s)} 条')

全部: 17898 条, 脉冲星=1639, RFI=16259
采样: 2000 条


In [3]:
# ===== PCA 降维到 2D + 缩放到 [0.15, 0.85] =====
X_2d = PCA(n_components=2).fit_transform(StandardScaler().fit_transform(X_s))
for j in range(2):
    lo, hi = X_2d[:,j].min(), X_2d[:,j].max()
    X_2d[:,j] = (X_2d[:,j] - lo) / (hi - lo) * 0.7 + 0.15

# ===== 9x9 均匀网格 =====
N = 9; h = 1.0 / (N - 1)
GX, GY = np.meshgrid(np.linspace(0,1,N), np.linspace(0,1,N), indexing='ij')
nodes_2d = np.column_stack([GX.ravel(), GY.ravel()])
boundary = (GX.ravel()==0) | (GX.ravel()==1) | (GY.ravel()==0) | (GY.ravel()==1)
internal = ~boundary
print(f'网格: {N}x{N}={N*N} 节点, 内部={internal.sum()}, QUBO变量={internal.sum()*BIT_WIDTH}')

网格: 9x9=81 节点, 内部=49, QUBO变量=392


In [4]:
# ===== 高斯核源项: f(node) = Σ y_k * exp(-||node - p_k||²/(2σ²)) =====
sigma = 0.12
f = np.zeros(N*N)
for idx_k in range(len(y_s)):
    dist2 = np.sum((nodes_2d - X_2d[idx_k])**2, axis=1)
    f += y_s[idx_k] * np.exp(-dist2 / (2*sigma**2))

# ===== 5点差分刚度矩阵 (内部节点) =====
idx_map = -np.ones(N*N, dtype=int)
idx_map[internal] = np.arange(internal.sum())
n_i = internal.sum()
K_ii = np.zeros((n_i, n_i))
for i in range(1, N-1):
    for j in range(1, N-1):
        k = i * N + j
        ki = idx_map[k]
        K_ii[ki, ki] = 4.0
        for di, dj in [(-1,0),(1,0),(0,-1),(0,1)]:
            ni, nj = i+di, j+dj
            nk = ni * N + nj
            if internal[nk]:
                K_ii[ki, idx_map[nk]] = -1.0

rhs = h**2 * f[internal]
cond_K = np.linalg.cond(K_ii)
print(f'K_ii: {K_ii.shape}, κ(K)={cond_K:.2e}, SPD={np.allclose(K_ii, K_ii.T)}')

K_ii: (49, 49), κ(K)=2.53e+01, SPD=True


In [5]:
# ===== 经典参考解 & 变量上下界 =====
u_ref = np.linalg.solve(K_ii, rhs)
margin = 0.5; rng_val = max(u_ref.max()-u_ref.min(), 0.1)
lb = u_ref.min() - margin * rng_val
ub = u_ref.max() + margin * rng_val
print(f'参考解: [{u_ref.min():.4f}, {u_ref.max():.4f}], 编码界: [{lb:.4f}, {ub:.4f}]')

参考解: [-30.3658, -1.0726], 编码界: [-45.0124, 13.5740]


In [6]:
# ===== 能量泛函 QUBO: E(u) = (1/2)u^T K_ii u - u^T rhs, Kronecker 构建 =====
def build_energy_qubo(K, r, bw, lb, ub):
    n = K.shape[0]; nvar = n * bw
    scale = (ub - lb) / (2**bw - 1)
    s = scale * np.array([2**k for k in range(bw)])
    ssT = np.outer(s, s); c = lb * np.ones(n); w = K @ c - r
    Q = np.zeros((nvar, nvar))
    for i in range(n):
        for j in range(i, n):
            aij = K[i,j]
            if abs(aij) < 1e-15: continue
            ri, rj = i*bw, j*bw
            blk = 0.5 * aij * ssT
            Q[ri:ri+bw, rj:rj+bw] += blk
            if i != j: Q[rj:rj+bw, ri:ri+bw] += blk.T
    for i in range(n):
        Q[i*bw:(i+1)*bw, i*bw:(i+1)*bw] += np.diag(w[i] * s)
    return Q.astype(np.float32), nvar, scale

Q_float, nvar, scale = build_energy_qubo(K_ii, rhs, BIT_WIDTH, lb, ub)
Q_qubo = kw.qubo.adjust_qubo_matrix_precision(Q_float)
print(f'QUBO: {nvar}x{nvar}, 值范围: [{Q_qubo.min():.1f}, {Q_qubo.max():.1f}]')

QUBO: 392x392, 值范围: [-508.0, 1268.0]


In [7]:
# ===== CIM 提交 (solve #1) =====
ising_mat, ising_bias = kw.conversion.qubo_matrix_to_ising_matrix(Q_qubo)
vars_ = [f'x[{i}]' for i in range(ising_mat.shape[0])]
ising_model = kw.ising.IsingModel(variables=vars_, ising_matrix=ising_mat, bias=ising_bias)
opt = kw.cim.CIMOptimizer(task_name='htru2_debug', task_mode='quota')
opt.solve(ising_model.get_matrix())
print(f'htru2_debug 已提交, Ising: {ising_mat.shape}')

[2026-05-21 20:40:25] [INFO    ] [kaiwu.cim._optimizer_adapter:6] - Task submit successfully, waiting for data validation. Task name: htru2_debug
htru2_debug 已提交, Ising: (393, 393)


In [9]:
# ===== CIM 取回 + 解码 (solve #2) =====
sol = opt.solve(ising_model.get_matrix())
print(f'返回: {sol.shape}')
sols_bin = (sol[:,:-1] * sol[:,-1:]+1)/2
energies = np.array([z@Q_qubo@z for z in sols_bin])
z_best = sols_bin[np.argmin(energies)]
u_quantum_i = np.zeros(n_i)
s = scale * np.array([2**k for k in range(BIT_WIDTH)])
for i in range(n_i):
    zi = z_best[i*BIT_WIDTH:(i+1)*BIT_WIDTH]
    u_quantum_i[i] = np.dot(s, zi) + lb
print(f'量子内部解: [{u_quantum_i.min():.4f}, {u_quantum_i.max():.4f}], 最优能量: {energies.min():.1f}')

[2026-05-21 20:42:12] [INFO    ] [kaiwu.cim._optimizer_adapter:2] - Task completed: htru2_debug
返回: (10, 393)
量子内部解: [-19.5101, 1.8567], 最优能量: -14072.0


In [10]:
# ===== 保存预设解 =====
u_quantum = np.zeros(N*N); u_quantum[internal] = u_quantum_i
np.save(f'{OUTPUT_DIR}/htru2_preset_nodes.npy', nodes_2d)
np.save(f'{OUTPUT_DIR}/htru2_preset_values.npy', u_quantum)
np.save(f'{OUTPUT_DIR}/htru2_X_2d.npy', X_2d)
np.save(f'{OUTPUT_DIR}/htru2_y.npy', y_s)
np.save(f'{OUTPUT_DIR}/htru2_K_ii.npy', K_ii)
np.save(f'{OUTPUT_DIR}/htru2_rhs.npy', rhs)
np.save(f'{OUTPUT_DIR}/htru2_internal_mask.npy', internal)

rmse = np.sqrt(np.mean((u_quantum_i - u_ref)**2))
meta = {
    'method': 'energy_qubo',
    'dataset': 'HTRU2_pulsar',
    'telescope': 'Parkes_64m_radio',
    'n_internal': int(n_i),
    'qubo_size': int(nvar),
    'bit_width': BIT_WIDTH,
    'kappa_K': float(cond_K),
    'rmse_vs_classical': float(rmse),
    'best_energy': float(energies.min())
}
with open(f'{OUTPUT_DIR}/meta_htru2_energy.json', 'w') as f:
    json.dump(meta, f, indent=2)
print(f'预设已保存, RMSE vs 经典: {rmse:.6e}')

预设已保存, RMSE vs 经典: 6.289913e+00
